# Submission v7c-alpha1p2 — Calibration α=1.2

v7c (α=1.5) scored LB **0.35441** (new best). Testing α=1.2 as a bracket.

| α | Class 0 | Class 1 | Class 2 | Notes |
|---|---------|---------|---------|-------|
| 1.2 | 31.7% | 12.9% | 55.3% | Lighter calibration |
| **1.5** | **25.8%** | **8.0%** | **66.2%** | **Current best LB** |
| 1.8 | 20.0% | 4.7% | 75.3% | Stronger calibration |

Train prior: 19.9% / 8.1% / 72.0%


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000; HALF_MS = 90_000; THIRD_MS = 60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k]=np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm)>=32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid=lrow['pid']; ts=float(lrow['timestamp']); lid=lrow['id']
        feat={'id':lid}
        sg=sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta=sg['timestamp'].values
        wa =sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf =sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl =sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1=sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3=sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]
        for c in SENSOR_COLS:
            v  =wa[c].dropna().values.astype(float)
            vf =wf[c].dropna().values.astype(float)
            vl =wl[c].dropna().values.astype(float)
            vt1=wt1[c].dropna().values.astype(float)
            vt3=wt3[c].dropna().values.astype(float)
            if len(v)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}']=np.nan
                continue
            feat[f'{c}_mean']   =float(np.mean(v));    feat[f'{c}_std']   =float(np.std(v))
            feat[f'{c}_min']    =float(np.min(v));     feat[f'{c}_max']   =float(np.max(v))
            feat[f'{c}_median'] =float(np.median(v))
            feat[f'{c}_skew']   =float(spstats.skew(v))     if len(v)>2 else 0.
            feat[f'{c}_kurt']   =float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[f'{c}_range']  =float(np.max(v)-np.min(v))
            feat[f'{c}_q25']    =float(np.percentile(v,25)); feat[f'{c}_q75']=float(np.percentile(v,75))
            feat[f'{c}_iqr']    =float(np.percentile(v,75)-np.percentile(v,25))
            feat[f'{c}_delta']  =float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[f'{c}_slope']  =float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[f'{c}_t1_mean']=float(np.mean(vt1)) if len(vt1)>0 else float(np.mean(v))
            feat[f'{c}_t3_mean']=float(np.mean(vt3)) if len(vt3)>0 else float(np.mean(v))
            feat[f'{c}_t3t1']   =feat[f'{c}_t3_mean']-feat[f'{c}_t1_mean']
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag=np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean']=float(np.mean(mag))
            feat['accel_mag_std'] =float(np.std(mag))
            feat['accel_mag_max'] =float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc']=pid_enc_map.get(pid,-1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  train_pid_map)
print(f'train: {train_features.shape}  test: {test_features.shape}')

tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']
imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),      columns=test_features.columns,  index=test_features.index)

counts=Counter(y); total=len(y); n_cls=len(counts)
class_weights={0:total/(n_cls*counts[0]), 1:min(total/(n_cls*counts[1]),2.5), 2:total/(n_cls*counts[2])}
sample_weights=np.array([class_weights[yi] for yi in y])
train_prior=np.array([counts[i]/total for i in range(3)])
print('Class weights:', {k:round(v,3) for k,v in class_weights.items()})
print('Train prior:', {i:round(train_prior[i],3) for i in range(3)})


Extracting features...
train: (815, 106)  test: (1028, 106)
Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior: {0: np.float64(0.199), 1: np.float64(0.081), 2: np.float64(0.72)}


In [3]:
LGBM_PARAMS=dict(n_estimators=1000,learning_rate=0.02,num_leaves=127,max_depth=-1,
    min_child_samples=5,subsample=0.6,colsample_bytree=0.6,reg_alpha=0.3,reg_lambda=0.3,
    class_weight='balanced',objective='multiclass',num_class=3,n_jobs=-1,verbose=-1)

SEEDS=[42,7,123]
all_test_proba=[]; all_cv_scores=[]
for seed in SEEDS:
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)
    seed_proba=np.zeros((len(X_test_imp),3)); fold_scores=[]
    for fold,(tr_idx,val_idx) in enumerate(skf.split(X_imp,y)):
        X_tr=X_imp.iloc[tr_idx]; y_tr=y.iloc[tr_idx]
        X_va=X_imp.iloc[val_idx]; y_va=y.iloc[val_idx]
        m=lgb.LGBMClassifier(**{**LGBM_PARAMS,'random_state':seed})
        m.fit(X_tr,y_tr,sample_weight=sample_weights[tr_idx],
              eval_set=[(X_va,y_va)],
              callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(-1)])
        sc=balanced_accuracy_score(y_va,m.predict(X_va))
        fold_scores.append(sc); seed_proba+=m.predict_proba(X_test_imp)
        print(f'  Seed {seed} Fold {fold+1}: val BA={sc:.4f}')
    seed_proba/=5; all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV={np.mean(fold_scores):.4f}')
print(f'Ensemble CV={np.mean(all_cv_scores):.4f}')
raw_proba=np.mean(all_test_proba,axis=0)

CALIB_ALPHA = 1.2
cal_proba = raw_proba * (train_prior ** CALIB_ALPHA)
cal_proba = cal_proba / cal_proba.sum(axis=1, keepdims=True)
final_preds = np.argmax(cal_proba, axis=1).astype(int)

print(f'\nCalibration alpha=1.2')
print('Raw distribution:')
for u,cnt in zip(*np.unique(np.argmax(raw_proba,1),return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Calibrated:')
for u,cnt in zip(*np.unique(final_preds,return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')
print('Train prior:')
for cls in range(3): print(f'  class {cls}: {counts[cls]}  ({counts[cls]/total*100:.1f}%)')

submission=pd.DataFrame({'id':TEST_LABEL['id'].values,'stress':final_preds})
submission.to_csv('submission_v7c_a12.csv',index=False)
print('\nsubmission_v7c_a12.csv saved!')
print(submission.head(10))


  Seed 42 Fold 1: val BA=0.8414
  Seed 42 Fold 2: val BA=0.7679
  Seed 42 Fold 3: val BA=0.8314
  Seed 42 Fold 4: val BA=0.7622
  Seed 42 Fold 5: val BA=0.8495
  Seed 42 mean CV=0.8105
  Seed 7 Fold 1: val BA=0.8179
  Seed 7 Fold 2: val BA=0.8094
  Seed 7 Fold 3: val BA=0.8327
  Seed 7 Fold 4: val BA=0.8341
  Seed 7 Fold 5: val BA=0.7982
  Seed 7 mean CV=0.8184
  Seed 123 Fold 1: val BA=0.8614
  Seed 123 Fold 2: val BA=0.6913
  Seed 123 Fold 3: val BA=0.8809
  Seed 123 Fold 4: val BA=0.8220
  Seed 123 Fold 5: val BA=0.7724
  Seed 123 mean CV=0.8056
Ensemble CV=0.8115

Calibration alpha=1.2
Raw distribution:
  class 0: 400  (38.9%)
  class 1: 479  (46.6%)
  class 2: 149  (14.5%)
Calibrated:
  class 0: 316  (30.7%)
  class 1: 119  (11.6%)
  class 2: 593  (57.7%)
Train prior:
  class 0: 162  (19.9%)
  class 1: 66  (8.1%)
  class 2: 587  (72.0%)

submission_v7c_a12.csv saved!
     id  stress
0  1227       2
1  1228       0
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  